In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from config import CONFIG

In [2]:
engine = create_engine(CONFIG["db_url"])
print("Connected")

Connected


In [3]:
df = pd.read_sql(
    f'SELECT * FROM "{CONFIG["schema"]}"."{CONFIG["clean_table"]}"',
    engine
)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded: 101,766 rows x 69 columns


In [4]:
# Target variable distribution
# Q1: What is the overall 30-day readmission rate?

target_dist = pd.read_sql("""
    SELECT
        readmitted,
        COUNT(*)                                             AS encounters,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2)  AS pct_of_total
    FROM public.vw_encounter_base
    GROUP BY readmitted
    ORDER BY encounters DESC
""", engine)

print("--- Target Variable Distribution ---")
print(target_dist.to_string(index=False))

--- Target Variable Distribution ---
readmitted  encounters  pct_of_total
        NO       52524         52.87
       >30       35502         35.74
       <30       11314         11.39


## Section 1: Target Variable Distribution
**Analytical Question Q1:** What is the overall 30-day readmission rate?

**Output:**

| readmitted | encounters | pct_of_total |
|---|---|---|
| NO | 52,524 | 52.87% |
| >30 | 35,502 | 35.74% |
| <30 | 11,314 | 11.39% |

**Total valid encounters: 99,340**
(101,766 total minus 2,423 expired/hospice and 3 invalid gender
rows excluded by vw_encounter_base clinical filters)

**Key Findings:**

1. **The overall 30-day readmission rate is 11.39%** — this is the
   headline KPI for the entire project and the baseline against which
   every segment rate will be compared. Any diagnosis category,
   specialty, or patient group showing a rate above 11.39% carries
   above-average readmission risk.

2. **Note on row count difference from profiling** — during profiling
   in notebook 01, the class imbalance check on the raw table showed
   11,357 rows with readmitted = '<30' and a rate of 11.16%. The
   current output shows 11,314 rows and 11.39%. This difference is
   expected and correct:
   - The profiling ran on the full raw table (101,766 rows)
   - This query runs on vw_encounter_base which excludes 2,423
     expired/hospice encounters and 3 invalid gender rows
   - Removing patients who cannot be readmitted from the denominator
     correctly increases the rate from 11.16% to 11.39%
   - 11.39% is the analytically correct rate to use in all KPIs

3. **More than half of encounters did not result in readmission**
   — 52.87% of patients were discharged and did not return within
   30 days or at any point in the study period. This is the majority
   class confirmed during profiling.

4. **>30 day readmissions are substantial at 35.74%** — more than
   one in three encounters led to a readmission after 30 days.
   This group is tracked separately in the base view as
   led_to_any_readmission and will appear in the LOS and
   utilization KPI views for comparison against the <30 group.

5. **Class imbalance confirmed in the valid population** —
   the ratio of NO to <30 is approximately 4.6:1 in the valid
   analysis population, consistent with the 4.8:1 ratio observed
   in the full raw table during profiling. The imbalance persists
   after clinical exclusions as expected.

**Baseline KPI established: 11.39%**
This number is referenced in every subsequent EDA section and
every KPI view. When a segment shows 14% the story is
"14% vs 11.39% baseline — 23% above average risk."
When a segment shows 9% the story is
"9% vs 11.39% baseline — 21% below average risk."
That framing is what makes findings meaningful rather than
just isolated numbers.

**Next:** Demographic EDA — which age, gender, and race groups
carry readmission rates above or below the 11.39% baseline?

In [5]:
# SECTION 2: Demographic EDA
# Q2:  Which demographics carry the highest risk?

# Age band readmission rates

age_dist =  pd.read_sql("""
SELECT
    age,
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct
FROM public.vw_encounter_base
GROUP BY age
ORDER BY age
""", engine)


print("---age band distribution---")
print(age_dist.to_string(index=False))

---age band distribution---
     age  encounters  readmitted_30day  readmission_rate_pct
  [0-10)         160                 3                  1.88
 [10-20)         690                40                  5.80
 [20-30)        1649               236                 14.31
 [30-40)        3764               424                 11.26
 [40-50)        9607              1024                 10.66
 [50-60)       17060              1667                  9.77
 [60-70)       22058              2493                 11.30
 [70-80)       25329              3054                 12.06
 [80-90)       16434              2065                 12.57
[90-100)        2589               308                 11.90


## Section 2a: Demographic EDA — Age Band Readmission Rates
**Analytical Question Q2:** Which demographics carry the highest
readmission risk?

**Baseline 30-day readmission rate: 11.39%**

**Output:**

| Age Band | Encounters | Readmitted 30-Day | Readmission Rate % |
|---|---|---|---|
| [0-10) | 160 | 3 | 1.88% |
| [10-20) | 690 | 40 | 5.80% |
| [20-30) | 1,649 | 236 | 14.31% |
| [30-40) | 3,764 | 424 | 11.26% |
| [40-50) | 9,607 | 1,024 | 10.66% |
| [50-60) | 17,060 | 1,667 | 9.77% |
| [60-70) | 22,058 | 2,493 | 11.30% |
| [70-80) | 25,329 | 3,054 | 12.06% |
| [80-90) | 16,434 | 2,065 | 12.57% |
| [90-100) | 2,589 | 308 | 11.90% |

**Key Findings:**

1. **[20-30) has the highest readmission rate at 14.31%** — this is
   the most surprising finding in the age analysis and is 2.92
   percentage points above the 11.39% baseline. Young adults aged
   20-30 are not typically considered high-risk for diabetes
   readmission but this rate suggests they may be presenting with
   more acute or poorly controlled diabetes, possibly Type 1
   diabetes which has earlier onset and higher management complexity.
   However this group has only 1,649 encounters — the smallest
   adult group — so this rate should be interpreted with caution
   as it may be influenced by a small number of complex cases.

2. **Older adults (70-90) consistently show above-baseline rates**
   — three consecutive age bands are above the 11.39% baseline:
   - [70-80): 12.06% — 0.67 points above baseline
   - [80-90): 12.57% — 1.18 points above baseline (highest among
     high-volume groups)
   - [90-100): 11.90% — 0.51 points above baseline
   This pattern is clinically consistent with the cumulative
   comorbidity burden that accumulates with age — older diabetic
   patients are more likely to have concurrent cardiovascular
   disease, kidney disease, and hypertension, all of which
   complicate recovery and increase readmission risk.

3. **[80-90) has the highest rate among high-volume age groups
   at 12.57%** — with 16,434 encounters this is a statistically
   reliable finding unlike the [20-30) group. Patients aged 80-90
   represent the highest-risk elderly segment and should be
   prioritized in any discharge planning or post-discharge
   intervention program.

4. **Middle-aged adults (40-60) show below-baseline rates**
   — both [40-50) at 10.66% and [50-60) at 9.77% are below the
   11.39% baseline. [50-60) has the lowest rate among all adult
   groups. This may reflect better disease management capability
   in working-age adults who have more active engagement with
   healthcare systems and stronger social support networks
   compared to elderly patients.

5. **[0-10) and [10-20) have very low rates at 1.88% and 5.80%**
   — pediatric and adolescent patients show the lowest readmission
   rates. However these groups have very small encounter counts
   (160 and 690 respectively) and are likely managed differently
   from adult diabetes patients — pediatric inpatient care
   typically involves more intensive post-discharge follow-up
   which may contribute to lower readmission rates.

6. **The relationship between age and readmission is not linear**
   — contrary to the simple assumption that older = higher risk,
   the data shows a U-shaped pattern where very young adults
   (20-30) and older adults (70-90) both show elevated rates
   while middle-aged adults (40-60) show the lowest rates.
   This non-linear relationship means age alone is not a
   sufficient predictor of readmission risk and must be
   combined with prior utilization and clinical complexity
   measures for meaningful risk stratification.

**Above Baseline (11.39%):**
- [20-30): 14.31% ↑ highest rate overall
- [80-90): 12.57% ↑ highest rate among high-volume groups
- [70-80): 12.06% ↑
- [90-100): 11.90% ↑

**Below Baseline (11.39%):**
- [30-40): 11.26% ↓
- [60-70): 11.30% ↓
- [40-50): 10.66% ↓
- [50-60): 9.77% ↓ lowest adult rate
- [10-20): 5.80% ↓
- [0-10): 1.88% ↓ lowest overall

**Clinical Implication:**
Age-based risk stratification should focus intervention resources
on two distinct groups — young adults aged 20-30 who may have
poorly controlled or newly diagnosed diabetes, and elderly patients
aged 70-90 who carry high comorbidity burden. Middle-aged patients
aged 40-60 represent the lowest-risk group and require less
intensive post-discharge monitoring.

**Next:** Gender readmission rates.

In [6]:
# Gender readmission rates

gender_readmission = pd.read_sql("""
SELECT
    gender,
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct
FROM public.vw_encounter_base
GROUP BY gender
ORDER BY readmission_rate_pct DESC
""", engine)

print("---Gender_readmission_rate---")
print(gender_readmission.to_string(index=False))


---Gender_readmission_rate---
gender  encounters  readmitted_30day  readmission_rate_pct
Female       53454              6128                 11.46
  Male       45886              5186                 11.30


## Section 2b: Demographic EDA — Gender Readmission Rates
**Baseline 30-day readmission rate: 11.39%**

**Output:**

| Gender | Encounters | Readmitted 30-Day | Readmission Rate % |
|---|---|---|---|
| Female | 53,454 | 6,128 | 11.46% |
| Male | 45,886 | 5,186 | 11.30% |

**Note:** Unknown/Invalid gender (3 rows) excluded by
vw_encounter_base clinical filter (gender_invalid_flag = 0)

**Key Findings:**

1. **Gender difference is minimal at 0.16 percentage points**
   — Female patients show a readmission rate of 11.46% and Male
   patients 11.30%. The gap between them is negligible and both
   rates sit very close to the 11.39% overall baseline. This
   suggests that gender alone is not a meaningful differentiator
   of readmission risk in this diabetes inpatient population.

2. **Female patients have slightly higher readmission rate**
   — at 11.46% females are 0.07 points above the baseline while
   males at 11.30% are 0.09 points below it. However this
   difference is so small that it is unlikely to be clinically
   meaningful. Statistical validation in notebook 05 will
   confirm whether this difference is statistically significant
   or attributable to random variation.

3. **Female patients represent the majority of encounters**
   — 53,454 female encounters (53.87%) vs 45,886 male encounters
   (46.13%), consistent with the gender distribution observed
   during profiling in Step 4 where Female was 53.76% and Male
   was 46.24% of the raw dataset. The proportions are preserved
   after clinical exclusions confirming the exclusion filters
   did not introduce gender bias.

4. **Both rates are very close to the 11.39% baseline**
   — Female is 0.07 points above and Male is 0.09 points below.
   Neither group deviates meaningfully from the overall population
   rate. This is a finding in itself — it tells us that the
   readmission risk observed in this dataset is distributed
   approximately equally across genders and that gender should
   not be a primary dimension for targeting readmission
   reduction interventions.

5. **Gender is not a key risk stratification variable**
   — unlike age where [80-90) showed 12.57% vs [50-60) at 9.77%
   (a gap of 2.80 percentage points), gender shows only a 0.16
   point gap. Clinicians and hospital administrators looking to
   identify high-risk patients should prioritize age, prior
   inpatient visits, and diagnosis category over gender as
   risk indicators.

**Above Baseline (11.39%):**
- Female: 11.46% ↑ (0.07 points above)

**Below Baseline (11.39%):**
- Male: 11.30% ↓ (0.09 points below)

**Clinical Implication:**
Gender does not meaningfully differentiate readmission risk in
this diabetes inpatient population. Readmission reduction
strategies should not be gender-targeted. Resources are better
directed toward age-based and utilization-based risk factors
identified in the age band analysis above.

**Statistical Note:**
The 0.16 percentage point difference between Female (11.46%)
and Male (11.30%) will be formally tested in notebook 05
using a chi-square test on the contingency table of gender
vs readmission status. Given the small magnitude of the
difference and the large sample sizes involved, the test
may reach statistical significance purely due to sample
size — a statistically significant result here would not
necessarily imply clinical significance.

**Next:** Race readmission rates.

In [7]:
# Rate readmission rate

race_readmission = pd.read_sql("""
SELECT
    race,
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct
FROM public.vw_encounter_base
WHERE race != 'Unknown'
GROUP BY race
ORDER BY readmission_rate_pct DESC
""", engine)

print("---Race readmission rate---")
print(race_readmission.to_string(index=False))


---Race readmission rate---
           race  encounters  readmitted_30day  readmission_rate_pct
      Caucasian       74220              8556                 11.53
Africanamerican       18772              2149                 11.45
       Hispanic        2017               212                 10.51
          Asian         628                65                 10.35
          Other        1471               144                  9.79


### 3. Demographic Analysis — Readmission Rate by Race

**Analytical Question:**  
Are there notable variations in 30-day readmission rates across different racial demographics?

| Race | Encounters | 30-Day Readmissions | Readmission Rate (%) |
| :--- | :--- | :--- | :--- |
| **Caucasian** | 74,220 | 8,556 | **11.53%** |
| **African American** | 18,772 | 2,149 | **11.45%** |
| **Hispanic** | 2,017 | 212 | **10.51%** |
| **Asian** | 628 | 65 | **10.35%** |
| **Other** | 1,471 | 144 | **9.79%** |

---

#### Descriptive Insights:
* **Baseline Drivers:** The two highest-volume groups, **Caucasian** (11.53%) and **African American** (11.45%), both track slightly above the overall baseline. Because they constitute roughly 93.6% of the entire dataset, their outcomes heavily dictate the population average.
* **Lower-Risk Segments:** **Hispanic** (10.51%), **Asian** (10.35%), and **Other** (9.79%) populations exhibit visibly lower readmission rates compared to the baseline.
* **Sample Size Context:** There is a 1.74% percentage point gap between the highest rate (Caucasian) and the lowest (Other). However, because the Hispanic, Asian, and Other demographics represent less than 7% of total encounters combined, these lower rates should be interpreted with caution due to smaller sample sizes.

In [8]:
# SECTION 3: CLINICAL UTILIZATION EDA
# Q7: Does length of stay or polypharmacy correlate readmission?
# Key clinical question: do readmitted patients show higher resource utilization during their index encounter?

polypharmacy = pd.read_sql("""
SELECT
    CASE
        WHEN readmitted = '<30' THEN '1 — Readmitted <30 Days'
        WHEN readmitted = '>30' THEN '2 — Readmitted >30 Days'
        ELSE                         '3 — Not Readmitted'
    END                                                         AS readmission_outcome,
    COUNT(*)                                                    AS encounters,
    ROUND(AVG(time_in_hospital), 2)                             AS avg_los_days,
    ROUND(AVG(num_medications), 2)                              AS avg_medications,
    ROUND(AVG(num_active_medications), 2)                       AS avg_active_medications,
    ROUND(AVG(num_lab_procedures), 2)                           AS avg_lab_procedures,
    ROUND(AVG(number_diagnoses), 2)                             AS avg_diagnoses,
    ROUND(AVG(total_prior_visits), 2)                           AS avg_prior_visits,
    ROUND(SUM(polypharmacy_flag) * 100.0 / COUNT(*), 2)        AS polypharmacy_pct
FROM public.vw_encounter_base
GROUP BY readmission_outcome
ORDER BY readmission_outcome
""", engine)

print("---Length of stay or polypharmacy---")
print(polypharmacy.to_string(index=False))



---Length of stay or polypharmacy---
    readmission_outcome  encounters  avg_los_days  avg_medications  avg_active_medications  avg_lab_procedures  avg_diagnoses  avg_prior_visits  polypharmacy_pct
1 — Readmitted <30 Days       11314          4.77            16.91                    1.19               44.22           7.69              2.02             84.06
2 — Readmitted >30 Days       35502          4.50            16.28                    1.22               43.84           7.65              1.62             82.80
     3 — Not Readmitted       52524          4.22            15.57                    1.17               42.00           7.17              0.74             76.86


### 4. Clinical Utilization & Polypharmacy Analysis

**Analytical Question:**  
How do resource utilization, hospital length of stay (LOS), and polypharmacy rates differ based on patient readmission outcomes?

| Readmission Outcome | Encounters | Avg LOS (Days) | Avg Medications | Avg Lab Procedures | Avg Diagnoses | Avg Prior Visits | Polypharmacy Rate (%) |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **1 — Readmitted <30 Days** | 11,314 | **4.77** | **16.91** | **44.22** | **7.69** | **2.02** | **84.06%** |
| **2 — Readmitted >30 Days** | 35,502 | 4.50 | 16.28 | 43.84 | 7.65 | 1.62 | 82.80% |
| **3 — Not Readmitted** | 52,524 | 4.22 | 15.57 | 42.00 | 7.17 | 0.74 | 76.86% |

---

#### Descriptive Insights:
* **Prior History as a Major Indicator:** The most dramatic variance across these groups is found in `avg_prior_visits`. Patients readmitted early (`<30`) average **2.02 prior visits**—nearly **three times higher** than patients who were not readmitted (0.74). 
* **Polypharmacy Prevalence:** Polypharmacy (high medication volume) is prevalent across this entire diabetic cohort, but it peaks severely at **84.06%** for the early readmission group. These patients manage more complex therapeutic regimens (`avg_medications` of 16.91), which naturally increases the risk of post-discharge complications.
* **Stepwise Severity Gradient:** There is a consistent, incremental increase across every single clinical metric as patient outcomes worsen. Early-readmitted patients stay in the hospital longer (4.77 days vs 4.22), undergo more lab testing (44.22 vs 42.00), and carry a heavier comorbidity burden (7.69 diagnoses vs 7.17).

In [9]:
# Length of stay distribution banded into clinical groupings

LOS = pd.read_sql("""
SELECT
    CASE
        WHEN time_in_hospital <= 2  THEN '1 — 1-2 days'
        WHEN time_in_hospital <= 5  THEN '2 — 3-5 days'
        WHEN time_in_hospital <= 7  THEN '3 — 6-7 days'
        WHEN time_in_hospital <= 14 THEN '4 — 8-14 days'
        ELSE                             '5 — 15+ days'
    END                                                         AS los_band,
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct
FROM public.vw_encounter_base
GROUP BY los_band
ORDER BY los_band
""",engine)

print("---Length of stay distribution---")
print(LOS.to_string(index=False))



---Length of stay distribution---
     los_band  encounters  readmitted_30day  readmission_rate_pct
 1 — 1-2 days       30713              2864                  9.33
 2 — 3-5 days       40865              4720                 11.55
 3 — 6-7 days       13051              1693                 12.97
4 — 8-14 days       14711              2037                 13.85


### 5. Clinical Analysis — Length of Stay (LOS) Band Distribution

**Analytical Question:**  
Does the duration of the initial hospital stay strongly correlate with 30-day readmission rates?

| Length of Stay Band | Encounters | 30-Day Readmissions | Readmission Rate (%) |
| :--- | :--- | :--- | :--- |
| **1 — 1-2 days** | 30,713 | 2,864 | **9.33%** |
| **2 — 3-5 days** | 40,865 | 4,720 | **11.55%** |
| **3 — 6-7 days** | 13,051 | 1,693 | **12.97%** |
| **4 — 8-14 days** | 14,711 | 2,037 | **13.85%** |

---

#### Descriptive Insights:
* **Strong Positive Correlation:** Unlike demographic variables, the length of stay bands show a definitive, progressive trend. As the time spent in the hospital increases, the subsequent 30-day readmission rate rises step-for-step.
* **The Short-Stay Baseline Drop:** Patients discharged quickly within 1–2 days exhibit a readmission rate of **9.33%** (2.06 percentage points *below* the baseline). This group likely represents lower-complexity cases or highly stable patients.
* **Extended Stay Escalation:** Extended hospitalizations show a sharp spike in risk, maxing out at **13.85%** for patients staying 8–14 days (2.46 percentage points *above* the baseline). 
* **Clinical Context:** A longer initial stay serves as an excellent proxy for underlying acute clinical complexity, severe complications, or multiple severe comorbidities, leaving the patient more fragile upon discharge.

In [10]:
# SECTION 4: DIAGNOSIS CATEGORY EDA 
# Q3:  Which diagnosis categories drive the most readmissions?

diagnosis_dist = pd.read_sql(r"""
SELECT
    CASE
        WHEN diag_1 ~ '^[Vv]'                              THEN 'Supplementary (V-codes)'
        WHEN diag_1 ~ '^[Ee]'                              THEN 'External Causes (E-codes)'
        WHEN diag_1 = 'None'                               THEN 'No Primary Diagnosis'
        WHEN diag_1 ~ '^\d+\.?\d*$'
             AND diag_1::NUMERIC BETWEEN 390  AND 459      THEN 'Circulatory'
        WHEN diag_1 ~ '^\d+\.?\d*$'
             AND diag_1::NUMERIC BETWEEN 460  AND 519      THEN 'Respiratory'
        WHEN diag_1 ~ '^\d+\.?\d*$'
             AND diag_1::NUMERIC BETWEEN 520  AND 579      THEN 'Digestive'
        WHEN diag_1 ~ '^\d+\.?\d*$'
             AND diag_1::NUMERIC BETWEEN 250  AND 250.99   THEN 'Diabetes'
        WHEN diag_1 ~ '^\d+\.?\d*$'
             AND diag_1::NUMERIC BETWEEN 800  AND 999      THEN 'Injury & Poisoning'
        WHEN diag_1 ~ '^\d+\.?\d*$'
             AND diag_1::NUMERIC BETWEEN 710  AND 739      THEN 'Musculoskeletal'
        WHEN diag_1 ~ '^\d+\.?\d*$'
             AND diag_1::NUMERIC BETWEEN 580  AND 629      THEN 'Genitourinary'
        WHEN diag_1 ~ '^\d+\.?\d*$'
             AND diag_1::NUMERIC BETWEEN 140  AND 239      THEN 'Neoplasms'
        WHEN diag_1 ~ '^\d+\.?\d*$'
             AND diag_1::NUMERIC BETWEEN 290  AND 319      THEN 'Mental Disorders'
        ELSE 'Other'
    END                                                         AS diagnosis_category,
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 2)                             AS avg_los_days
FROM public.vw_encounter_base
GROUP BY diagnosis_category
ORDER BY readmission_rate_pct DESC
""",engine)

print("---Diagnosis Category Distribution---")
print(diagnosis_dist.to_string(index=False))




---Diagnosis Category Distribution---
       diagnosis_category  encounters  readmitted_30day  readmission_rate_pct  avg_los_days
  Supplementary (V-codes)        1632               265                 16.24          7.53
                 Diabetes        8661              1135                 13.10          4.33
       Injury & Poisoning        6851               850                 12.41          4.62
         Mental Disorders        2256               276                 12.23          5.87
              Circulatory       29584              3459                 11.69          4.22
              Respiratory        9919              1112                 11.21          5.03
            Genitourinary        4963               546                 11.00          4.24
                Neoplasms        3131               341                 10.89          5.28
                Digestive        9072               961                 10.59          4.39
                    Other       18335     

**Technical Note:**
SQL strings containing regex patterns (backslash sequences such as
\d and \.) are written as Python raw strings using the r prefix:
r"""...""". This tells Python to treat backslashes as literal
characters rather than escape sequences, ensuring the regex
pattern is passed to PostgreSQL exactly as written without
modification by the Python interpreter.

## Section 4: Diagnosis Category EDA
**Analytical Question Q3:** Which primary diagnosis categories
drive the most readmissions?
**Baseline 30-day readmission rate: 11.39%**

**Output:**

| Diagnosis Category | Encounters | Readmitted 30-Day | Rate % | Avg LOS |
|---|---|---|---|---|
| Supplementary (V-codes) | 1,632 | 265 | 16.24% | 7.53 days |
| Diabetes | 8,661 | 1,135 | 13.10% | 4.33 days |
| Injury & Poisoning | 6,851 | 850 | 12.41% | 4.62 days |
| Mental Disorders | 2,256 | 276 | 12.23% | 5.87 days |
| Circulatory | 29,584 | 3,459 | 11.69% | 4.22 days |
| Respiratory | 9,919 | 1,112 | 11.21% | 5.03 days |
| Genitourinary | 4,963 | 546 | 11.00% | 4.24 days |
| Neoplasms | 3,131 | 341 | 10.89% | 5.28 days |
| Digestive | 9,072 | 961 | 10.59% | 4.39 days |
| Other | 18,335 | 1,898 | 10.35% | 3.77 days |
| Musculoskeletal | 4,935 | 471 | 9.54% | 3.91 days |
| External Causes (E-codes) | 1 | 0 | 0.00% | 5.00 days |

**Key Findings:**

1. **Supplementary V-codes have the highest readmission rate at
   16.24%** — 4.85 percentage points above the 11.39% baseline.
   V-codes represent encounters coded for reasons other than active
   disease — such as follow-up care, screenings, or administrative
   reasons. A 16.24% readmission rate in this group suggests these
   patients are returning to hospital frequently despite the
   apparently routine nature of their visit. This is clinically
   significant — it may indicate that V-code encounters are being
   used for patients who are already medically fragile and
   returning for ongoing management of underlying conditions.
   The average LOS of 7.53 days is the highest of all categories,
   further confirming high clinical complexity in this group.

2. **Diabetes as primary diagnosis shows 13.10%** — 1.71 points
   above baseline. This is notable because the entire dataset
   consists of diabetic patients, yet only 8,661 encounters have
   diabetes as the primary diagnosis. These are patients where
   diabetes itself — rather than a comorbidity — drove the
   admission. Their above-baseline readmission rate of 13.10%
   confirms that poorly controlled diabetes as the primary
   presenting condition carries elevated readmission risk.

3. **Circulatory disease is the highest volume category**
   — 29,584 encounters representing 29.77% of all valid
   encounters. Despite the high volume, the readmission rate
   of 11.69% is only 0.30 points above baseline — moderate
   elevation. However in absolute terms Circulatory disease
   produces 3,459 readmissions which is the highest raw count
   of any category. From a hospital resource perspective,
   even a small rate reduction in Circulatory disease would
   yield the greatest absolute reduction in readmissions.

4. **Mental Disorders show 12.23% despite low volume**
   — 2,256 encounters but 0.84 points above baseline. Psychiatric
   comorbidities in diabetic patients are known to complicate
   self-management of blood glucose, medication adherence, and
   dietary control — all of which contribute to higher readmission
   risk. This group warrants targeted post-discharge mental health
   support alongside diabetes management.

5. **Musculoskeletal has the lowest rate at 9.54%** — 1.85 points
   below baseline. Musculoskeletal conditions such as joint
   disorders and fractures have more defined treatment pathways
   and clearer discharge criteria than metabolic or cardiovascular
   conditions, which may explain the lower readmission rate.
   Average LOS of 3.91 days is among the lowest, suggesting
   efficient care delivery for this category.

6. **External Causes (E-codes) has only 1 encounter** — the
   0.00% readmission rate is not analytically meaningful due
   to the single encounter. This category will be excluded
   from the KPI view using the HAVING COUNT(*) >= 50 threshold
   to ensure all reported rates are statistically reliable.

7. **Relationship between LOS and readmission rate is not linear**
   — V-codes have both the highest LOS (7.53 days) and the highest
   readmission rate (16.24%). But Respiratory has the second
   highest LOS (5.03 days) yet only 11.21% readmission rate.
   Musculoskeletal has one of the lowest LOS (3.91 days) and
   also the lowest readmission rate (9.54%). This suggests that
   LOS alone does not determine readmission risk — the underlying
   diagnosis category and patient complexity are stronger drivers.

**Above Baseline (11.39%):**
- Supplementary V-codes : 16.24% ↑ (+4.85 points) — highest rate
- Diabetes              : 13.10% ↑ (+1.71 points)
- Injury & Poisoning    : 12.41% ↑ (+1.02 points)
- Mental Disorders      : 12.23% ↑ (+0.84 points)
- Circulatory           : 11.69% ↑ (+0.30 points) — highest volume

**Below Baseline (11.39%):**
- Respiratory           : 11.21% ↓ (-0.18 points)
- Genitourinary         : 11.00% ↓ (-0.39 points)
- Neoplasms             : 10.89% ↓ (-0.50 points)
- Digestive             : 10.59% ↓ (-0.80 points)
- Other                 : 10.35% ↓ (-1.04 points)
- Musculoskeletal       :  9.54% ↓ (-1.85 points) — lowest rate

**Clinical Implication:**
Two distinct intervention priorities emerge from this analysis:

Priority 1 — High rate, manageable volume:
V-codes (16.24%), Diabetes (13.10%), and Mental Disorders (12.23%)
show the highest rates. Targeted post-discharge protocols for
these groups could meaningfully reduce readmission rates even
with relatively small encounter volumes.

Priority 2 — Moderate rate, very high volume:
Circulatory disease at 11.69% across 29,584 encounters produces
the most absolute readmissions (3,459). Even reducing the
Circulatory rate by 1 percentage point would prevent approximately
296 readmissions — more impact than eliminating readmissions
entirely in the Musculoskeletal category.

**Next:** Medical specialty and admission source EDA.

In [11]:
# SECTION 5: MEDICAL SPECIALTY AND ADMISSION SOURCE EDA
# Which clinical departments have thehighest readmission rates?

medical_specialty = pd.read_sql("""
SELECT
    medical_specialty,
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 2)                             AS avg_los_days
FROM public.vw_encounter_base
WHERE medical_specialty != 'Unknown'
GROUP BY medical_specialty
HAVING COUNT(*) >= 50
ORDER BY encounters DESC
LIMIT 15
""",engine)

print("---Medical specialty readmission rate---")
print(medical_specialty.to_string(index=False))

---Medical specialty readmission rate---
              medical_specialty  encounters  readmitted_30day  readmission_rate_pct  avg_los_days
               Internalmedicine       14237              1642                 11.53          4.57
               Emergency/Trauma        7419               845                 11.39          4.36
         Family/Generalpractice        7252               879                 12.12          4.35
                     Cardiology        5278               424                  8.03          3.50
                Surgery-General        3059               342                 11.18          4.53
                     Nephrology        1539               248                 16.11          5.04
                    Orthopedics        1392               151                 10.85          3.98
     Orthopedics-Reconstructive        1230                92                  7.48          3.91
                    Radiologist        1121               102                

## Section 5a: Medical Specialty EDA
**Analytical Question Q4:** Which clinical departments have the
highest readmission rates?
**Baseline 30-day readmission rate: 11.39%**
**Filter applied:** Specialties with >= 50 encounters only
Unknown specialty excluded (~49% of encounters)

**Output:**

| Medical Specialty | Encounters | Readmitted 30-Day | Rate % | Avg LOS |
|---|---|---|---|---|
| Internal Medicine | 14,237 | 1,642 | 11.53% | 4.57 days |
| Emergency/Trauma | 7,419 | 845 | 11.39% | 4.36 days |
| Family/General Practice | 7,252 | 879 | 12.12% | 4.35 days |
| Cardiology | 5,278 | 424 | 8.03% | 3.50 days |
| Surgery-General | 3,059 | 342 | 11.18% | 4.53 days |
| Nephrology | 1,539 | 248 | 16.11% | 5.04 days |
| Orthopedics | 1,392 | 151 | 10.85% | 3.98 days |
| Orthopedics-Reconstructive | 1,230 | 92 | 7.48% | 3.91 days |
| Radiologist | 1,121 | 102 | 9.10% | 3.45 days |
| Pulmonology | 854 | 96 | 11.24% | 5.21 days |
| Psychiatry | 853 | 104 | 12.19% | 6.40 days |
| Urology | 682 | 67 | 9.82% | 3.41 days |
| Obstetrics and Gynecology | 669 | 32 | 4.78% | 3.10 days |
| Surgery-Cardiovascular/Thoracic | 642 | 43 | 6.70% | 5.77 days |
| Gastroenterology | 538 | 61 | 11.34% | 4.48 days |

**Key Findings:**

1. **Nephrology has the highest readmission rate at 16.11%**
   — 4.72 percentage points above the 11.39% baseline and the
   highest rate of any specialty with meaningful encounter volume.
   Nephrology patients are diabetic patients with concurrent
   kidney disease — a well-documented high-risk combination.
   Diabetes is the leading cause of chronic kidney disease
   globally and patients with both conditions have severely
   limited physiological reserve. Poor glycemic control
   accelerates kidney damage, and deteriorating kidney function
   impairs medication metabolism, making diabetes management
   harder. The 5.04 day average LOS further confirms the
   clinical complexity of this patient group. Nephrology
   should be a primary target for enhanced post-discharge
   follow-up protocols.

2. **Family/General Practice shows 12.12%** — 0.73 points above
   baseline despite being a generalist setting. This is
   counterintuitive at first — general practice admissions
   might be expected to involve less complex cases than
   specialist departments. However this rate may reflect
   that general practice admissions capture patients whose
   conditions did not clearly fit a specialist pathway,
   potentially including patients with multiple comorbidities
   managed across several systems simultaneously, which
   increases readmission risk.

3. **Psychiatry shows 12.19% with the highest average LOS
   among outpatient-linked specialties at 6.40 days**
   — consistent with the Mental Disorders diagnosis category
   finding in Section 4 (12.23%). Psychiatric comorbidities
   in diabetic patients impair medication adherence, dietary
   management, and glucose monitoring — all direct drivers
   of readmission. The high LOS of 6.40 days suggests these
   patients require longer stabilization periods yet still
   leave at above-baseline readmission risk.

4. **Internal Medicine is the highest volume specialty
   at 14,237 encounters** — nearly double the next largest
   specialty (Emergency/Trauma at 7,419). At 11.53% its
   readmission rate is only 0.14 points above baseline but
   in absolute terms it produces 1,642 readmissions — the
   highest raw count of any specialty. Similar to the
   Circulatory diagnosis finding, even marginal improvement
   in Internal Medicine readmission rates would yield the
   greatest absolute reduction in hospital readmissions.

5. **Emergency/Trauma exactly matches the baseline at 11.39%**
   — a precise alignment with the overall population rate.
   This is notable because emergency admissions represent
   unplanned acute deterioration which would typically be
   expected to carry higher readmission risk. The exact
   baseline match may reflect that emergency stabilization
   followed by appropriate discharge planning produces
   outcomes equivalent to the broader population.

6. **Cardiology has a surprisingly low rate at 8.03%**
   — 3.36 points below baseline despite managing
   cardiovascular disease which is a major diabetes comorbidity.
   This may reflect that cardiology departments have well
   established, protocol-driven discharge processes including
   cardiac rehabilitation referrals and structured medication
   management that reduce readmission risk effectively.
   The lowest average LOS of 3.50 days suggests efficient
   care delivery rather than premature discharge.

7. **Obstetrics and Gynecology has the lowest rate at 4.78%**
   — 6.61 points below baseline. This is expected — OB/GYN
   admissions in a diabetic dataset likely represent
   gestational diabetes management or pregnancy-related
   complications which have defined treatment endpoints
   and structured post-natal follow-up protocols that
   naturally reduce readmission risk.

8. **Surgery-Cardiovascular/Thoracic shows 6.70% despite
   high LOS of 5.77 days** — cardiovascular surgery patients
   typically receive intensive post-operative monitoring and
   structured cardiac rehabilitation. The high LOS reflects
   surgical complexity but the low readmission rate suggests
   that thorough in-hospital recovery before discharge
   reduces the need to return.

9. **Orthopedics-Reconstructive at 7.48% is notably lower
   than standard Orthopedics at 10.85%** — reconstructive
   procedures follow highly structured rehabilitation
   protocols with defined recovery milestones. Standard
   orthopedics may involve more acute fracture management
   where discharge timing is less controlled.

**Above Baseline (11.39%):**
- Nephrology              : 16.11% ↑ (+4.72 points) — highest rate
- Psychiatry              : 12.19% ↑ (+0.80 points)
- Family/General Practice : 12.12% ↑ (+0.73 points)
- Internal Medicine       : 11.53% ↑ (+0.14 points) — highest volume
- Emergency/Trauma        : 11.39% = (exactly at baseline)

**Below Baseline (11.39%):**
- Surgery-General         : 11.18% ↓ (-0.21 points)
- Pulmonology             : 11.24% ↓ (-0.15 points)
- Gastroenterology        : 11.34% ↓ (-0.05 points)
- Orthopedics             : 10.85% ↓ (-0.54 points)
- Urology                 :  9.82% ↓ (-1.57 points)
- Radiologist             :  9.10% ↓ (-2.29 points)
- Cardiology              :  8.03% ↓ (-3.36 points)
- Surgery-Cardiovascular  :  6.70% ↓ (-4.69 points)
- Orthopedics-Reconstruct.:  7.48% ↓ (-3.91 points)
- OB/GYN                  :  4.78% ↓ (-6.61 points) — lowest rate

**Clinical Implication:**
Three specialties require priority attention for readmission
reduction interventions:

**Immediate priority — Nephrology (16.11%):**
The combination of diabetes and kidney disease creates the
highest-risk patient profile in this dataset. Enhanced
post-discharge nephrology follow-up, dietitian referrals
for renal diabetic diets, and medication reconciliation
at discharge should be prioritized for this group.

**Volume priority — Internal Medicine (14,237 encounters):**
Even reducing Internal Medicine readmission rate by 1
percentage point from 11.53% to 10.53% would prevent
approximately 142 readmissions — more than eliminating
all readmissions in the Urology or Gastroenterology
specialties entirely.

**Comorbidity priority — Psychiatry (12.19%):**
Psychiatric comorbidities consistently appear as a
readmission risk factor across both the diagnosis category
analysis and the specialty analysis. Integrated diabetes
and mental health management during and after admission
represents an opportunity to reduce readmission risk
in this complex patient group.

**Next:** Admission source readmission rates.

In [12]:
# Admission source readmission rates

admission_source = pd.read_sql("""
SELECT
    admission_source_id,
    CASE admission_source_id
        WHEN '1' THEN 'Physician Referral'
        WHEN '2' THEN 'Clinic Referral'
        WHEN '3' THEN 'HMO Referral'
        WHEN '4' THEN 'Transfer from Hospital'
        WHEN '5' THEN 'Transfer from SNF'
        WHEN '6' THEN 'Transfer from Another Facility'
        WHEN '7' THEN 'Emergency Room'
        WHEN '8' THEN 'Court / Law Enforcement'
        WHEN '9' THEN 'Not Available'
        ELSE          'Other'
    END                                                         AS source_label,
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct
FROM public.vw_encounter_base
GROUP BY admission_source_id
ORDER BY readmission_rate_pct DESC
""",engine)

print("---Admission source readmission---")
print(admission_source.to_string(index=False))



---Admission source readmission---
admission_source_id                   source_label  encounters  readmitted_30day  readmission_rate_pct
                 22                          Other          12                 2                 16.67
                  3                   HMO Referral         185                29                 15.68
                 20                          Other         159                22                 13.84
                  8        Court / Law Enforcement          15                 2                 13.33
                  5              Transfer from SNF         806               100                 12.41
                  7                 Emergency Room       55848              6690                 11.98
                 17                          Other        6570               704                 10.72
                  1             Physician Referral       29167              3120                 10.70
                  9                  N

## Section 5b: Admission Source EDA
**Analytical Question Q8:** Does admission source affect
readmission risk?
**Baseline 30-day readmission rate: 11.39%**

**Output:**

| Source ID | Source Label | Encounters | Readmitted 30-Day | Rate % |
|---|---|---|---|---|
| 22 | Other | 12 | 2 | 16.67% |
| 3 | HMO Referral | 185 | 29 | 15.68% |
| 20 | Other | 159 | 22 | 13.84% |
| 8 | Court/Law Enforcement | 15 | 2 | 13.33% |
| 5 | Transfer from SNF | 806 | 100 | 12.41% |
| 7 | Emergency Room | 55,848 | 6,690 | 11.98% |
| 17 | Other | 6,570 | 704 | 10.72% |
| 1 | Physician Referral | 29,167 | 3,120 | 10.70% |
| 9 | Not Available | 125 | 13 | 10.40% |
| 2 | Clinic Referral | 1,081 | 111 | 10.27% |
| 4 | Transfer from Hospital | 3,118 | 309 | 9.91% |
| 6 | Transfer from Another Facility | 2,239 | 212 | 9.47% |
| 10,11,13,14,25 | Other (various) | 15 combined | 0 | 0.00% |

**Key Findings:**

1. **Emergency Room is by far the dominant admission source**
   — 55,848 encounters representing 56.22% of all valid
   encounters. More than half of all diabetic inpatient
   admissions in this dataset came through the emergency
   room. At 11.98% readmission rate it sits 0.59 points
   above the 11.39% baseline. In absolute terms Emergency
   Room admissions produce 6,690 readmissions — the highest
   raw count of any source by a very wide margin. Even a
   small reduction in the Emergency Room readmission rate
   would have the largest absolute impact on total
   readmissions across the hospital system.

2. **HMO Referral shows the highest meaningful rate at 15.68%**
   — 4.29 points above baseline among sources with more than
   100 encounters. HMO (Health Maintenance Organization)
   referrals represent patients whose insurance plan directed
   them to specific facilities. The high readmission rate
   may reflect that HMO-referred patients are being
   transferred between facilities with gaps in care
   continuity and information handover, increasing the
   risk of post-discharge complications requiring return.

3. **Transfer from SNF (Skilled Nursing Facility) shows 12.41%**
   — 1.02 points above baseline. Patients transferred from
   skilled nursing facilities are already receiving
   post-acute care, meaning they are recovering from a
   prior illness or procedure. Being admitted from an SNF
   indicates a deterioration in condition despite ongoing
   care — a profile associated with higher clinical
   complexity and readmission risk. This group warrants
   enhanced discharge coordination back to the SNF.

4. **Physician Referral is the second largest source
   at 29,167 encounters (29.37%)** — at 10.70% its
   readmission rate is 0.69 points below baseline.
   Planned physician referrals represent scheduled
   admissions where the clinical need was identified
   in advance, allowing for better preparation and
   more structured discharge planning. The

In [13]:
# SECTION 6: MEDICATION AND DIABETES MANAGEMENT EDA
# Q5 Does medication management correlate withreadmission risk?

#Insulin prescription and readmission

insulin_presc = pd.read_sql("""
SELECT
    insulin,
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct
FROM public.vw_encounter_base
GROUP BY insulin
ORDER BY readmission_rate_pct DESC
""",engine)

print("---Insulin prescription and readmission---")
print(insulin_presc.to_string(index=False))



---Insulin prescription and readmission---
insulin  encounters  readmitted_30day  readmission_rate_pct
   Down       11908              1693                 14.22
     Up       10987              1465                 13.33
 Steady       30069              3420                 11.37
     No       46376              4736                 10.21


## Section 6a: Medication EDA — Insulin Prescription
**Analytical Question Q5:** Does medication management correlate
with readmission risk?
**Baseline 30-day readmission rate: 11.39%**

**Output:**

| Insulin Status | Encounters | Readmitted 30-Day | Rate % |
|---|---|---|---|
| Down (dose decreased) | 11,908 | 1,693 | 14.22% |
| Up (dose increased) | 10,987 | 1,465 | 13.33% |
| Steady (dose unchanged) | 30,069 | 3,420 | 11.37% |
| No (not prescribed) | 46,376 | 4,736 | 10.21% |

**Total encounters: 99,340 — all valid encounters accounted for**

**Key Findings:**

1. **Patients with insulin dose decreased (Down) have the highest
   readmission rate at 14.22%** — 2.83 points above baseline.
   A downward dose adjustment during admission suggests the
   patient arrived with hypoglycemia or dangerously low blood
   glucose, or that their condition improved enough during
   the stay to require less insulin. Either scenario indicates
   an unstable glycemic state at the time of the encounter
   which is associated with higher post-discharge readmission
   risk. These patients may be discharged before full
   glycemic stability is achieved.

2. **Patients with insulin dose increased (Up) show 13.33%**
   — 1.94 points above baseline. An upward adjustment
   indicates the patient arrived with hyperglycemia or
   poorly controlled blood glucose requiring more aggressive
   insulin management during the stay. The elevated
   readmission rate suggests that increasing insulin during
   admission does not fully resolve the underlying glycemic
   instability by the time of discharge.

3. **Both dose change groups (Down and Up) are above baseline**
   — any insulin dose adjustment during the encounter,
   whether upward or downward, is associated with above-
   baseline readmission rates. This is a clinically important
   combined finding: insulin dose adjustment is a signal of
   glycemic instability, and glycemic instability at
   admission predicts readmission risk regardless of the
   direction of adjustment.

4. **Steady insulin patients exactly match the baseline
   at 11.37%** — patients whose insulin dose remained
   unchanged during the encounter show a rate virtually
   identical to the 11.39% overall baseline (0.02 points
   below). This makes clinical sense — unchanged dosing
   indicates relative glycemic stability during the
   admission, which is associated with average readmission
   risk rather than elevated risk.

5. **Patients not prescribed insulin have the lowest rate
   at 10.21%** — 1.18 points below baseline. These are
   diabetic patients managing their condition without
   insulin, typically through oral medications or diet
   and lifestyle alone. They represent patients with
   less severe or less advanced diabetes — Type 2
   patients who have not yet required insulin therapy.
   Their lower readmission rate reflects lower disease
   severity rather than better management per se.

6. **Clear dose-response pattern observed:**

In [14]:
#  A1C result and readmission
# None means the test was NOT performed — not a normal result

A1Cresult = pd.read_sql("""
SELECT
    "A1Cresult",
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct
FROM public.vw_encounter_base
GROUP BY "A1Cresult"
ORDER BY readmission_rate_pct DESC
""",engine)

print("---AIC result and readmission---")
print(A1Cresult.to_string(index=False))



---AIC result and readmission---
A1Cresult  encounters  readmitted_30day  readmission_rate_pct
     None       82506              9641                 11.69
       >7        3775               383                 10.15
       >8        8137               809                  9.94
     Norm        4922               481                  9.77


## Section 6b: Medication EDA — A1C Result and Readmission
**Analytical Question Q5 continued:** Does glycemic control
as measured by HbA1c testing correlate with readmission risk?
**Baseline 30-day readmission rate: 11.39%**

**Critical note on A1C result values:**
As documented in the data dictionary, "None" in this column
means the A1C test was NOT performed during the encounter —
it does not mean the result was normal. This distinction is
clinically critical and fundamentally changes the interpretation
of this output.

**Output:**

| A1C Result | Encounters | Readmitted 30-Day | Rate % |
|---|---|---|---|
| None (not tested) | 82,506 | 9,641 | 11.69% |
| >7 (poor control) | 3,775 | 383 | 10.15% |
| >8 (very poor control) | 8,137 | 809 | 9.94% |
| Norm (good control) | 4,922 | 481 | 9.77% |

**Total encounters: 99,340 — all valid encounters accounted for**

**Key Findings:**

1. **83% of encounters had no A1C test performed** — 82,506
   out of 99,340 valid encounters (83.06%) have A1Cresult =
   None, meaning the HbA1c test was not ordered during the
   admission. This is the single most important finding in
   this section. In a dataset consisting entirely of diabetic
   inpatient encounters, having no A1C measurement in 83%
   of cases represents a significant gap in diabetes
   monitoring practice across these 130 hospitals during
   the 1999-2008 study period.

2. **Untested patients show the highest readmission rate
   at 11.69%** — 0.30 points above the 11.39% baseline.
   Patients who were not tested for A1C during their
   admission show higher readmission rates than all
   three tested groups. This supports the clinical
   hypothesis that A1C testing during admission is
   associated with better diabetes management and
   consequently lower readmission risk — potentially
   because ordering an A1C test reflects a more thorough
   approach to diabetes care during the encounter.

3. **All three tested groups show below-baseline rates**
   — regardless of whether the A1C result indicated good
   or poor glycemic control, patients who had the test
   performed all showed lower readmission rates than
   patients who were not tested:
   - >7  (poor control)      : 10.15% — 1.24 points below baseline
   - >8  (very poor control) : 9.94%  — 1.45 points below baseline
   - Norm (good control)     : 9.77%  — 1.62 points below baseline

4. **Counterintuitive finding — very poor control (>8)
   shows lower rate than poor control (>7)** — patients
   with A1C > 8 (very poor glycemic control) show 9.94%
   while patients with A1C > 7 show 10.15%. This small
   difference of 0.21 points is unlikely to be clinically
   meaningful but the direction is unexpected. One
   possible explanation is that patients with very poor
   control (>8) may receive more intensive management
   and closer follow-up during and after admission
   precisely because their condition is more severe,
   which paradoxically reduces readmission risk.

5. **The tested vs untested gap is more meaningful than
   the differences within tested groups** — the 1.92
   percentage point gap between untested (11.69%) and
   best-performing tested group Norm (9.77%) is far
   larger than the 0.38 point spread across the three
   tested groups (10.15% to 9.77%). This suggests that
   the act of testing itself — rather than the result
   of the test — is the stronger predictor of lower
   readmission risk. A1C testing may be a proxy for
   overall quality of diabetes care during admission.

6. **Small sample sizes in tested groups require caution**
   — the three tested groups combined account for only
   16,834 encounters (16.94% of valid encounters).
   Norm has only 4,922 encounters and >7 has only 3,775.
   The readmission rate differences across these groups
   should be interpreted carefully. Statistical validation
   in notebook 05 will confirm whether these differences
   are statistically significant or within the range of
   random variation given the sample sizes.

**Above Baseline (11.39%):**
- None (not tested) : 11.69% ↑ (+0.30 points) — 83% of encounters

**Below Baseline (11.39%):**
- >7  (poor control)      : 10.15% ↓ (-1.24 points)
- >8  (very poor control) :  9.94% ↓ (-1.45 points)
- Norm (good control)     :  9.77% ↓ (-1.62 points) — lowest rate

**The Unexpected Pattern:**

The expected clinical hierarchy — where worse glycemic
control should produce higher readmission rates — does
not hold within the tested group. All tested patients
perform better than untested patients regardless of
their result. This finding challenges the assumption
that A1C result level is the primary driver of
readmission risk and instead suggests that testing
practice itself is the more important variable.

**Clinical Implication:**
This analysis produces one of the most actionable findings
in the entire project. The data suggests that simply
ordering an A1C test during a diabetic inpatient admission
is associated with lower 30-day readmission rates —
regardless of what the result shows.

Two possible mechanisms explain this:
1. **Selection effect** — clinicians who order A1C tests
   provide more comprehensive diabetes care overall,
   and it is this broader care quality rather than
   the test itself that reduces readmission risk
2. **Direct effect** — A1C results inform discharge
   medication adjustments and follow-up planning,
   directly improving post-discharge diabetes management

Either mechanism supports the same policy recommendation:
universal A1C testing for diabetic inpatient admissions
should be considered as a standard of care — not as
an optional add-on — particularly given that only 17%
of encounters in this 1999-2008 dataset had the test
performed.

**Next:** Medication change and diabetes medication cross-tab.

In [15]:
#  Medication change cross-tab
#change = Ch means at least one diabetes medication dose was adjusted during the encounter — a sign of activeclinical management diabetesmed = Yes means patient was on diabetes medication
#The most important comparison is Ch+Yes vs No+Yes: does active dose adjustment reduce readmission risk?

medication_change = pd.read_sql("""
SELECT
    change                                                      AS medication_change,
    "diabetesMed"                                                 AS on_diabetes_medication,
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct
FROM public.vw_encounter_base
GROUP BY change, "diabetesMed"
ORDER BY readmission_rate_pct DESC
""",engine)

print("---Medication change cross-tab---")
print(medication_change.to_string(index=False))



---Medication change cross-tab---
medication_change on_diabetes_medication  encounters  readmitted_30day  readmission_rate_pct
               Ch                    Yes       46120              5545                 12.02
               No                    Yes       30597              3536                 11.56
               No                     No       22623              2233                  9.87


## Section 6c: Medication EDA — Medication Change Cross-Tab
**Analytical Question Q5 continued:** Does active medication
management during the encounter correlate with readmission risk?
**Baseline 30-day readmission rate: 11.39%**

**Output:**

| Medication Change | On Diabetes Medication | Encounters | Readmitted 30-Day | Rate % |
|---|---|---|---|---|
| Ch (changed) | Yes | 46,120 | 5,545 | 12.02% |
| No (unchanged) | Yes | 30,597 | 3,536 | 11.56% |
| No (unchanged) | No | 22,623 | 2,233 | 9.87% |

**Note on missing combination:**
The combination Ch + No (medication changed but not on diabetes
medication) does not appear in the output. This is logically
correct — a patient cannot have a diabetes medication dose
changed if they are not prescribed diabetes medication.
The cross-tab is therefore complete with three valid combinations.

**Total encounters: 99,340 — all valid encounters accounted for**

**Key Findings:**

1. **Patients with medication changes show the highest rate
   at 12.02%** — 0.63 points above the 11.39% baseline.
   Ch + Yes represents patients actively prescribed diabetes
   medication whose dose was adjusted during the encounter.
   As established in the insulin analysis in Section 6a,
   dose adjustment is a signal of glycemic instability
   at the time of admission. The above-baseline rate
   here is consistent with that finding and extends it
   beyond insulin to all diabetes medications combined.

2. **Patients on diabetes medication with no dose change
   show 11.56%** — 0.17 points above baseline. No + Yes
   represents patients prescribed diabetes medication
   whose doses remained stable throughout the admission
   indicating relative glycemic stability. The near-
   baseline rate confirms that medication stability
   during admission is associated with average readmission
   risk — consistent with the Steady insulin finding
   in Section 6a which also showed a near-baseline rate.

3. **Patients not on diabetes medication show the lowest
   rate at 9.87%** — 1.52 points below baseline. No + No
   represents patients managing diabetes without any
   prescribed diabetes medication — likely through diet,
   exercise, and lifestyle modification alone. These
   patients have the mildest form of diabetes and the
   lowest clinical complexity, which explains the
   below-baseline readmission rate. This group is
   consistent with the No insulin group in Section 6a
   which also showed the lowest rate at 10.21%.

4. **A clear three-tier risk pattern emerges:**

Each tier is separated by approximately 1.0 to 1.2
   percentage points, creating a consistent stepped
   pattern where increasing medication involvement
   corresponds to increasing readmission risk. This
   reflects increasing diabetes severity across the
   three groups rather than a failure of medication
   management per se.

5. **Medication change group is the largest at 46,120
   encounters (46.43%)** — nearly half of all valid
   encounters involved patients whose diabetes medication
   was actively adjusted during the admission. This is
   a striking finding — the majority of diabetic inpatient
   admissions in this dataset involved active medication
   titration, confirming the high clinical complexity
   of this population.

6. **Combined reading with insulin analysis:**
   Both the insulin-specific analysis (Section 6a) and
   this combined medication analysis point to the same
   conclusion — any active medication adjustment during
   admission signals glycemic instability and predicts
   above-baseline readmission risk. The consistency
   across both analyses strengthens the reliability
   of this finding.

**Above Baseline (11.39%):**
- Ch + Yes : 12.02% ↑ (+0.63 points) — medication adjusted
- No + Yes : 11.56% ↑ (+0.17 points) — medication stable

**Below Baseline (11.39%):**
- No + No  :  9.87% ↓ (-1.52 points) — no medication prescribed

**Clinical Implication:**
The 2.15 percentage point gap between patients with
medication changes (12.02%) and patients not on diabetes
medication (9.87%) reflects the spectrum of diabetes
severity in this inpatient population rather than
a management quality issue.

The most actionable finding is within the Ch + Yes group:
46,120 encounters involving medication adjustment at
12.02% readmission rate. These patients are identifiable
at the point of discharge because their medication was
changed during the admission. They represent an ideal
target for enhanced post-discharge follow-up — the
medication change is a flag that is already known to
the clinical team at discharge and requires no
additional screening to identify.

A post-discharge protocol specifically targeting patients
with diabetes medication changes — such as a pharmacist
telephone review at 3 and 7 days post-discharge —
could directly address the glycemic instability
that this analysis identifies as the primary driver
of elevated readmission risk in this group.

**Next:** Polypharmacy and readmission rates.

In [16]:
#Polypharmacy vs readmission
#Polypharmacy threshold: >= 10 medications (clinical standard)
# 80% of encounters involve polypharmacy patients as found during feature engineering
#this query tests whether the medication burden itself correlates with readmission

polypharmacy_read = pd.read_sql("""
SELECT
    CASE WHEN polypharmacy_flag = 1 THEN 'Polypharmacy (10+ meds)'
         ELSE 'Standard (<10 meds)' END                        AS medication_burden,
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct
FROM public.vw_encounter_base
GROUP BY polypharmacy_flag
ORDER BY readmission_rate_pct DESC
""",engine)

print("--Polypharmacy vs readmission rate---")
print(polypharmacy_read.to_string(index=False))

--Polypharmacy vs readmission rate---
      medication_burden  encounters  readmitted_30day  readmission_rate_pct
Polypharmacy (10+ meds)       79274              9511                 12.00
    Standard (<10 meds)       20066              1803                  8.99


## Section 6d: Medication EDA — Polypharmacy and Readmission
**Analytical Question Q5 continued:** Does medication burden
correlate with readmission risk?
**Baseline 30-day readmission rate: 11.39%**
**Polypharmacy threshold: >= 10 medications (clinical standard)**

**Output:**

| Medication Burden | Encounters | Readmitted 30-Day | Rate % |
|---|---|---|---|
| Polypharmacy (10+ meds) | 79,274 | 9,511 | 12.00% |
| Standard (<10 meds) | 20,066 | 1,803 | 8.99% |

**Key Findings:**

1. **Polypharmacy patients show 12.00% vs standard 8.99%**
   — a gap of 3.01 percentage points. This is one of the
   largest rate differences observed across any binary
   split in this entire EDA. Patients on 10 or more
   medications are 33.5% more likely to be readmitted
   within 30 days than patients on fewer than 10
   medications. The polypharmacy threshold of 10
   medications is a clinically established standard
   and this data confirms its relevance as a readmission
   risk indicator in this diabetic inpatient population.

2. **Polypharmacy affects 79.81% of all valid encounters**
   — 79,274 out of 99,340 encounters involve patients on
   10 or more medications. This was first identified during
   feature engineering in notebook 03 where the polypharmacy
   flag mean of 0.80 indicated 80% prevalence. The EDA
   confirms this figure on the valid analysis population.
   Nearly 4 in 5 diabetic inpatient encounters in this
   dataset involve polypharmacy patients — reflecting the
   high comorbidity burden typical of this population.

3. **Polypharmacy patients are 0.61 points above baseline**
   — at 12.00% the polypharmacy group sits 0.61 points
   above the 11.39% baseline. Given that polypharmacy
   patients represent 79.81% of the population they
   effectively set the population rate. The overall
   11.39% baseline is heavily influenced by the 12.00%
   polypharmacy rate simply because this group dominates
   the encounter count.

4. **Standard medication patients are 2.40 points below
   baseline at 8.99%** — the 20,066 encounters involving
   patients on fewer than 10 medications show the lowest
   readmission rate observed across all medication-related
   analyses in this EDA. This group represents patients
   with lower disease complexity — fewer concurrent
   conditions requiring medication management means
   simpler post-discharge self-management and lower
   readmission risk.

5. **The 3.01 point gap is clinically and statistically
   meaningful** — across a combined population of 99,340
   encounters this difference represents approximately
   3,000 excess readmissions attributable to the
   polypharmacy group compared to what would be expected
   if they had the same rate as the standard group.
   Statistical validation in notebook 05 using chi-square
   testing will confirm whether this difference reaches
   statistical significance — given the large sample
   sizes involved it almost certainly will.

6. **Consistency across medication analyses:**
   All four medication-related analyses in Section 6
   point to the same underlying pattern — greater
   medication involvement and complexity is associated
   with higher readmission risk:

   | Analysis | Lower Risk Group | Higher Risk Group | Gap |
   |---|---|---|---|
   | Insulin | No (10.21%) | Down (14.22%) | 4.01 pts |
   | A1C | Tested (9.77-10.15%) | Not tested (11.69%) | 1.54-1.92 pts |
   | Med change | No medication (9.87%) | Changed (12.02%) | 2.15 pts |
   | Polypharmacy | Standard (8.99%) | Polypharmacy (12.00%) | 3.01 pts |

   The consistency of this pattern across four independent
   analyses strengthens confidence that medication complexity
   is a genuine readmission risk signal rather than a
   statistical artifact.

**Above Baseline (11.39%):**
- Polypharmacy (10+ meds) : 12.00% ↑ (+0.61 points)
  — affects 79.81% of all encounters

**Below Baseline (11.39%):**
- Standard (<10 meds) : 8.99% ↓ (-2.40 points)
  — affects 20.19% of all encounters

**Magnitude of Difference:**
Polypharmacy patients are readmitted at a rate 33.5% higher
than standard medication patients:
(12.00 - 8.99) / 8.99 × 100 = 33.5% relatively higher risk

**Clinical Implication:**
Polypharmacy is both highly prevalent (79.81% of encounters)
and associated with meaningfully elevated readmission risk
(12.00% vs 8.99%). This combination makes it one of the
most important risk factors identified in this analysis.

Three specific polypharmacy-related readmission risks
are clinically recognized and supported by this data:

1. **Medication adherence complexity** — patients on 10
   or more medications face significant challenges
   managing their regimen correctly after discharge,
   particularly elderly patients (70-90 age group
   identified as high-risk in Section 2a) who may
   have cognitive or physical barriers to adherence

2. **Drug interaction risk** — higher medication counts
   increase the probability of adverse drug interactions
   that cause post-discharge complications requiring
   hospital return

3. **Discharge counseling burden** — adequately counseling
   a patient on 10+ medications at discharge requires
   more time and resources than the average discharge
   process typically allows, leading to gaps in patient
   understanding of their regimen

**Recommendation for KPI dashboard:**
Polypharmacy flag should be included as a filter in the
Power BI dashboard so clinicians can toggle between the
polypharmacy and standard populations across all KPI
views — enabling drill-through from the high-risk segment
page to see how polypharmacy interacts with diagnosis
category, specialty, and age band simultaneously.

**This completes Section 6 — Medication and Diabetes
Management EDA.**

Key finding summary for Section 6:
- Insulin dose adjustment (any direction) → above baseline
- A1C not tested → above baseline (83% of encounters)
- Medication changed during admission → above baseline
- Polypharmacy (10+ meds) → above baseline (80% of encounters)
- Patients not on diabetes medication → furthest below baseline
- Patients on standard medication burden → furthest below baseline

All four analyses consistently show that medication complexity
and instability are associated with elevated readmission risk
while medication simplicity is associated with reduced risk.

**Next:** Repeat patient and prior utilization EDA.

In [17]:
# SECTION 7: REPEAT PATIENT AND PRIOR UTILIZATION EDA
# Do prior visits predict readmission?
# Do repeat patients have higher readmission rates?

#  Encounter sequence vs readmission risk

encounter_tier = pd.read_sql("""
SELECT
    CASE
        WHEN encounter_seq = 1 THEN '1st Encounter'
        WHEN encounter_seq = 2 THEN '2nd Encounter'
        WHEN encounter_seq = 3 THEN '3rd Encounter'
        ELSE                        '4th+ Encounter'
    END                                                         AS encounter_tier,
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct
FROM public.vw_encounter_base
GROUP BY encounter_tier
ORDER BY readmission_rate_pct DESC
""",engine)

print("---Encounter sequence vs readmission risk---")
print(encounter_tier.to_string(index=False))

---Encounter sequence vs readmission risk---
encounter_tier  encounters  readmitted_30day  readmission_rate_pct
4th+ Encounter        6889              1647                 23.91
 3rd Encounter        6123              1068                 17.44
 2nd Encounter       16341              2314                 14.16
 1st Encounter       69987              6285                  8.98


## Section 7a: Repeat Patient EDA — Encounter Sequence
**Analytical Question Q6:** Do prior visits predict readmission?
**Analytical Question Q9:** Do repeat patients have higher rates?
**Baseline 30-day readmission rate: 11.39%**

**Output:**

| Encounter Tier | Encounters | Readmitted 30-Day | Rate % |
|---|---|---|---|
| 4th+ Encounter | 6,889 | 1,647 | 23.91% |
| 3rd Encounter | 6,123 | 1,068 | 17.44% |
| 2nd Encounter | 16,341 | 2,314 | 14.16% |
| 1st Encounter | 69,987 | 6,285 | 8.98% |

**Key Findings:**

1. **The most striking finding in the entire EDA —
   a perfect monotonic escalation in readmission risk
   with each successive encounter:**
   1st encounter : 8.98%   ← 2.41 points BELOW baseline
2nd encounter : 14.16%  ← 2.77 points ABOVE baseline
3rd encounter : 17.44%  ← 6.05 points ABOVE baseline
4th+ encounter: 23.91%  ← 12.52 points ABOVE baseline

Every additional hospital encounter is associated with
   a higher readmission rate than the one before it.
   This is not a coincidence — it reflects genuine clinical
   escalation. Each readmission makes the next one more
   likely by further destabilizing glycemic control,
   introducing new complications, and weakening the
   patient's overall physiological reserve.

2. **4th+ encounter patients show 23.91%** — more than
   double the 11.39% baseline and the highest readmission
   rate of any segment identified across the entire EDA,
   surpassing even Nephrology (16.11%) and V-codes (16.24%).
   These 6,889 encounters represent patients who have
   been admitted to hospital at least 4 times and are
   returning at an alarming rate of nearly 1 in 4
   admissions. They represent the highest-risk patient
   segment in the entire dataset.

3. **3rd encounter patients show 17.44%** — 6.05 points
   above baseline. The jump from 2nd encounter (14.16%)
   to 3rd encounter (17.44%) is 3.28 percentage points,
   the largest single-step increase in the sequence.
   This suggests that by the third admission the patient's
   condition has significantly deteriorated and the
   cycle of readmission has become self-reinforcing —
   each hospitalization causes further deconditioning,
   increased medication complexity, and reduced capacity
   for self-management.

4. **2nd encounter patients show 14.16%** — 2.77 points
   above baseline. The transition from first to second
   encounter represents a 5.18 percentage point jump
   (8.98% to 14.16%) — the largest absolute increase
   in the sequence. The second admission is the critical
   inflection point where a patient transitions from
   the low-risk first-timer group to the elevated-risk
   repeat group. This makes the second encounter the
   most important intervention opportunity — preventing
   a second admission is the highest-leverage action
   a hospital can take to reduce the overall readmission
   rate.

5. **1st encounter patients show 8.98%** — 2.41 points
   below baseline. First-time patients represent 70.46%
   of all valid encounters (69,987 out of 99,340) and
   they drive the overall rate toward the baseline.
   Their below-baseline rate of 8.98% confirms that
   patients presenting to hospital for the first time
   in this dataset have not yet entered the readmission
   cycle and carry lower risk than the overall population.

6. **This finding validates the LAG() window function
   logic in vw_encounter_base** — the perfectly ordered
   monotonic escalation across encounter tiers confirms
   that the ROW_NUMBER() OVER (PARTITION BY patient_nbr
   ORDER BY encounter_id) window function is correctly
   sequencing encounters per patient. If the window
   function were broken, the pattern would be random
   rather than ordered. The clean escalation is both
   a clinical finding and a technical validation.

7. **Encounter distribution is heavily skewed toward
   first encounters:**
   - 1st encounter : 69,987 (70.46% of valid encounters)
   - 2nd encounter : 16,341 (16.45%)
   - 3rd encounter :  6,123  (6.16%)
   - 4th+ encounter:  6,889  (6.93%)
   The majority of encounters are first encounters with
   a rapidly diminishing tail of repeat encounters.
   Despite representing only 6.93% of encounters the
   4th+ group produces 1,647 readmissions — a
   disproportionate readmission burden relative to
   their encounter share.

8. **Readmission burden analysis:**
   

In [18]:
# Repeat vs first-time patient comparison
# This query quantifies whether repeat patients drive disproportionately higher readmission rates

patient_type = pd.read_sql("""
SELECT
    CASE WHEN is_repeat_patient = 1 THEN 'Repeat Patient'
         ELSE 'First-Time Patient' END                         AS patient_type,
    COUNT(*)                                                    AS encounters,
    SUM(led_to_30day_readmission)                               AS readmitted_30day,
    ROUND(SUM(led_to_30day_readmission) * 100.0
          / COUNT(*), 2)                                        AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 2)                             AS avg_los,
    ROUND(AVG(num_medications), 2)                              AS avg_medications,
    ROUND(AVG(total_prior_visits), 2)                           AS avg_prior_visits
FROM public.vw_encounter_base
GROUP BY patient_type
""",engine)

print("--- Repeat vs first-time patient comparison---")
print(patient_type.to_string(index=False))

--- Repeat vs first-time patient comparison---
      patient_type  encounters  readmitted_30day  readmission_rate_pct  avg_los  avg_medications  avg_prior_visits
First-Time Patient       53646              2287                  4.26     4.20            15.58              0.53
    Repeat Patient       45694              9027                 19.76     4.59            16.45              1.99


## Section 7b: Repeat Patient EDA — Repeat vs First-Time
**Analytical Question Q9 continued:** Do repeat patients have
significantly higher readmission rates than first-time patients?
**Baseline 30-day readmission rate: 11.39%**

**Output:**

| Patient Type | Encounters | Readmitted 30-Day | Rate % | Avg LOS | Avg Medications | Avg Prior Visits |
|---|---|---|---|---|---|---|
| First-Time Patient | 53,646 | 2,287 | 4.26% | 4.20 days | 15.58 | 0.53 |
| Repeat Patient | 45,694 | 9,027 | 19.76% | 4.59 days | 16.45 | 1.99 |

**Key Findings:**

1. **The single largest rate gap in the entire EDA —
   19.76% vs 4.26%, a difference of 15.50 percentage
   points.** Repeat patients are readmitted within 30
   days at a rate 4.64 times higher than first-time
   patients:
   19.76 / 4.26 = 4.64x higher relative risk
   This is not a marginal difference — it represents
   a fundamentally different clinical profile between
   the two groups. Repeat patients are not simply
   sicker versions of first-time patients — they
   represent a qualitatively different patient
   population with established patterns of frequent
   hospitalization.

2. **First-time patients show 4.26%** — 7.13 points
   below the 11.39% baseline. This is the lowest
   readmission rate of any meaningful segment
   identified across the entire EDA, lower even
   than OB/GYN (4.78%) and Musculoskeletal (9.54%).
   First-time patients represent patients who have
   not yet entered the hospital readmission cycle —
   their very low rate confirms that initial
   presentations to this hospital system, even for
   complex diabetic conditions, do not typically
   lead to immediate readmission.

3. **Repeat patients show 19.76%** — 8.37 points
   above baseline and the second highest rate of
   any segment in the EDA after 4th+ encounter
   patients (23.91%). Nearly 1 in 5 repeat patient
   encounters leads to a readmission within 30 days.
   The 45,694 repeat patient encounters produce
   9,027 readmissions which is 73.7% of all 30-day
   readmissions in the dataset despite representing
   only 45.99% of encounters:

   First-time patients : 53,646 encounters (53.99%)
2,287 readmissions (18.5% of total)
Repeat patients     : 45,694 encounters (45.99%)
9,027 readmissions (73.0% of total)

Repeat patients generate 73% of all readmissions
   while accounting for 46% of encounters — a profound
   disproportionality that defines the readmission
   problem in this dataset.

4. **Average LOS difference is small but meaningful**
   — repeat patients average 4.59 days vs 4.20 days
   for first-time patients, a difference of 0.39 days.
   The LOS difference is much smaller than the
   readmission rate difference (0.39 days vs 15.50
   percentage points), confirming that repeat patients
   are not simply staying longer — they are being
   discharged at similar lengths of stay but returning
   more frequently. This suggests the issue is post-
   discharge management rather than in-hospital care.

5. **Average medications difference is modest**
   — repeat patients average 16.45 medications vs
   15.58 for first-time patients, a difference of
   0.87 medications. Both groups are well above the
   polypharmacy threshold of 10 medications. The
   small difference confirms that polypharmacy is
   pervasive across both groups — medication burden
   alone does not explain the 15.50 point readmission
   rate gap between them.

6. **Average prior visits difference is substantial**
   — repeat patients average 1.99 prior healthcare
   visits vs 0.53 for first-time patients, a
   difference of 1.46 visits. Despite being a
   relatively small absolute difference, repeat
   patients have 3.75 times more prior healthcare
   contacts than first-time patients:
   1.99 / 0.53 = 3.75x more prior visits
   This confirms that repeat patients are high
   utilizers of the healthcare system not just
   within the current encounter but across their
   entire prior healthcare history.

7. **Clarification on encounter counts vs patient counts:**
   - Unique patients in dataset    : 71,518
   - Patients with 1 encounter     : 54,745 (first-time)
   - Patients with 2+ encounters   : 16,773 (repeat)
   The encounter counts in this output (53,646 and
   45,694) represent encounters attributed to each
   patient type — not unique patients. Repeat patients
   contribute multiple encounters each, which is why
   45,694 encounters come from only 16,773 unique
   patients (average of 2.72 encounters per repeat
   patient). This distinction is important — a small
   number of patients generates a large proportion
   of repeat encounters and readmissions.

8. **Connection to encounter sequence finding:**
   This output is the aggregate view of what Section
   7a showed at the granular level. First-time patients
   in this view correspond to 1st encounter patients
   in Section 7a (8.98% — note the rate differs because
   Section 7a used encounter sequence from vw_encounter_base
   while this view uses is_repeat_patient flag which
   classifies based on whether any repeat encounter
   exists for that patient). Repeat patients aggregate
   the 2nd, 3rd, and 4th+ encounter groups from
   Section 7a into one combined group (19.76%).

**Above Baseline (11.39%):**
- Repeat Patient     : 19.76% ↑ (+8.37 points)
  — 45.99% of encounters, 73.0% of all readmissions

**Below Baseline (11.39%):**
- First-Time Patient :  4.26% ↓ (-7.13 points)
  — 53.99% of encounters, 18.5% of all readmissions

**The Disproportionality:**
Patient type       Encounter share    Readmission share
First-time         53.99%             18.5%
Repeat             45.99%             73.0%

Repeat patients generate 3.95x their proportional
share of readmissions relative to their encounter count.
This extreme disproportionality is the defining
characteristic of the readmission problem in this dataset
and the strongest argument for targeting interventions
specifically at repeat patients rather than the
general inpatient population.

**Utilization Profile Comparison:**

Metric               First-Time    Repeat    Difference
Readmission rate     4.26%         19.76%    +15.50 pts
Avg LOS              4.20 days     4.59 days +0.39 days
Avg medications      15.58         16.45     +0.87 meds
Avg prior visits     0.53          1.99      +1.46 visits
Relative risk        1.00x         4.64x     —

The striking observation from this comparison is that
the utilization metrics (LOS, medications, prior visits)
show modest differences between the two groups while
the readmission rate shows an extreme difference.
This decoupling suggests that what separates repeat
from first-time patients is not captured by these
utilization metrics alone — it likely reflects
underlying disease trajectory, social determinants
of health, and care continuity factors that are
not present in this dataset.

**Clinical Implication:**
This analysis answers Q9 definitively and provides
the strongest single finding in the entire EDA:

**73% of all 30-day readmissions are generated by
45.99% of encounters — those belonging to repeat
patients.**

This concentration of readmission risk in a
identifiable, known subgroup (patients who have
been admitted before) is highly actionable because:

1. **These patients are known at the point of admission**
   — their prior encounter history is in the hospital
   system. No predictive model is needed to identify
   them — they self-identify through their admission
   history.

2. **The intervention window is clear** — the 30-day
   readmission window begins at discharge. Intensive
   post-discharge support in the first 7-14 days
   targets the period of highest readmission risk.

3. **The potential impact is large** — if a targeted
   intervention reduced the repeat patient readmission
   rate from 19.76% to the first-time patient rate
   of 4.26% (an aspirational but illustrative target),
   it would prevent approximately 7,073 readmissions:
   45,694 × (19.76% - 4.26%) = 7,082 prevented readmissions
   This would reduce the overall 30-day readmission
   rate from 11.39% to approximately 4.14% —
   a transformation of the hospital's readmission profile.

**This completes Section 7 — Repeat Patient and
Prior Utilization EDA.**

Key findings summary for Section 7:
- Readmission risk escalates perfectly with each
  successive encounter: 8.98% → 14.16% → 17.44% → 23.91%
- The 1st to 2nd encounter transition is the highest-
  leverage intervention point (+5.18 percentage points)
- Repeat patients generate 73% of all readmissions
  despite representing 46% of encounters
- The gap between repeat (19.76%) and first-time (4.26%)
  patients is the largest binary split in the entire EDA
- LOS and medication differences between groups are
  modest — the readmission rate difference is extreme

**This completes the SQL EDA phase.**
All 7 analytical questions have been addressed through
SQL queries on vw_encounter_base. The next phase is
statistical validation in notebook 05 to confirm
which EDA findings are statistically significant
versus attributable to random variation.

**Next Phase:** 05_statistical_validation.ipynb